In [ ]:
import gtfs2nx as gx
import folium
import osmnx as ox
import networkx as nx
import branca.colormap as cm
import pandas as pd
import zipfile

%matplotlib inline

In [ ]:
def gen_graph(scale:str, city:str, start_time:str = '07:00', end_time:str = '10:00'):
    path = f'../Data/{scale}/{city}.zip'
    G = gx.transit_graph(path, time_window=(start_time, end_time))

    print(f"Graph generated with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G

In [ ]:
def visualize_graph(G: nx.DiGraph, scale:str, city:str):
    """
    Takes a gtfs2nx graph and the path to its original GTFS zip file,
    and returns an interactive Folium map colored by transport mode and travel time.
    """
    gtfs_path = f'../Data/{scale}/{city}.zip'
    # 1. Convert to MultiDiGraph and Project
    G_multi = nx.MultiDiGraph(G)
    if 'crs' not in G_multi.graph:
        G_multi.graph['crs'] = "EPSG:32632"
    G_mapping = ox.project_graph(G_multi, to_crs="EPSG:4326")

    # --- THE BULLETPROOF GTFS DICTIONARY BUILDER ---
    with zipfile.ZipFile(gtfs_path, 'r') as z:
        routes = pd.read_csv(z.open('routes.txt'), dtype=str)
        stops = pd.read_csv(z.open('stops.txt'), dtype=str)

    routes['route_type'] = pd.to_numeric(routes['route_type'], errors='coerce')

    # Create two incredibly simple master dictionaries
    stop_names = stops.set_index('stop_id')['stop_name'].to_dict()
    route_types = routes.set_index('route_id')['route_type'].to_dict()

    def get_mode_color(rtype):
        if 100 <= rtype < 200: return 'red'       # Heavy Rail / Train
        if 400 <= rtype < 500: return 'purple'    # Urban Rail / Metro
        if 900 <= rtype < 1000: return 'orange'   # Tram
        if 700 <= rtype < 800: return 'blue'      # Bus
        return 'gray'

    # --- ANALYTICS LOGIC ---
    travel_times = [data.get('weight', 0) for u, v, data in G_mapping.edges(data=True) if data.get('weight', 0) > 0]
    if travel_times:
        min_time, max_time = min(travel_times), max(travel_times)
    else:
        min_time, max_time = 0, 1

    time_colormap = cm.LinearColormap(colors=['#00b050', '#ffff00', '#ff0000'], vmin=min_time, vmax=max_time)
    time_colormap.caption = 'Travel Time between stops (seconds)'

    # --- DRAWING THE MAP ---
    lats = [data.get('y') for node, data in G_mapping.nodes(data=True) if data.get('y') is not None]
    lons = [data.get('x') for node, data in G_mapping.nodes(data=True) if data.get('x') is not None]

    # Safety check in case the graph is empty
    if not lats or not lons:
        print("Graph contains no valid coordinates.")
        return None

    center_lat, center_lon = sum(lats) / len(lats), sum(lons) / len(lons)

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB DarkMatter')
    if travel_times:
        m.add_child(time_colormap)

    # 2. Draw the Edges
    for u, v, data in G_mapping.edges(data=True):
        u_lat, u_lon = G_mapping.nodes[u].get('y'), G_mapping.nodes[u].get('x')
        v_lat, v_lon = G_mapping.nodes[v].get('y'), G_mapping.nodes[v].get('x')

        # Extract the route_id from the edge's nodes to color the transit link!
        u_route_id = str(u).split('@@')[1] if '@@' in str(u) else None
        u_type = route_types.get(u_route_id, 700)
        edge_mode_color = get_mode_color(u_type)

        edge_time = data.get('weight', 0)
        line_color = time_colormap(edge_time) if edge_time > 0 else edge_mode_color

        if u != v and u_lat and u_lon and v_lat and v_lon:
            folium.PolyLine(
                locations=[(u_lat, u_lon), (v_lat, v_lon)],
                color=line_color,
                weight=2.5 if edge_mode_color in ['red', 'purple'] else 1.0,
                opacity=0.8,
                tooltip=f"Travel Time: {int(edge_time)}s" if edge_time > 0 else "Transit Link"
            ).add_to(m)

    # 3. Draw the Nodes
    for node, data in G_mapping.nodes(data=True):
        lat, lon = data.get('y'), data.get('x')
        node_str = str(node)

        # Parse the gtfs2nx node format: stop_id@@route_id
        if '@@' in node_str:
            stop_id_part, route_id_part = node_str.split('@@', 1)
        else:
            stop_id_part, route_id_part = node_str, None

        # Look up true name and type using the split pieces!
        name = stop_names.get(stop_id_part, stop_id_part)
        rtype = route_types.get(route_id_part, 700)
        node_color = get_mode_color(rtype)

        mode_labels = {'red': 'Train', 'purple': 'Metro', 'orange': 'Tram', 'blue': 'Bus', 'gray': 'Other'}
        mode_label = mode_labels.get(node_color, "Unknown")
        if lat and lon:
            folium.CircleMarker(
                location=(lat, lon),
                radius=1.5 if node_color in ['red', 'purple'] else 1,
                color=node_color,
                fill=True,
                fill_color=node_color,
                fill_opacity=1.0,
                tooltip=f"{name} ({mode_label})"
            ).add_to(m)

    # Return the map object so Jupyter can display it!
    return m

In [ ]:
g = gen_graph('agglomeration', 'bern')
m = visualize_graph(g, 'agglomeration', 'bern')
m

In [ ]:
g = gen_graph('agglomeration', 'fribourg')
m = visualize_graph(g, 'agglomeration', 'fribourg')
m

In [ ]:
g = gen_graph('agglomeration', 'lausanne')
m = visualize_graph(g, 'agglomeration', 'lausanne')
m

In [ ]:
g = gen_graph('agglomeration', 'zurich')
m = visualize_graph(g, 'agglomeration', 'zurich')
m

In [ ]:
g = gen_graph('canton', 'lu')
m = visualize_graph(g, 'canton', 'lu')
m

In [ ]:
g = gen_graph('district', 'bern')
m = visualize_graph(g, 'district', 'bern')
m